# 05 · Modellvergleich & Analyse — Favorita

Vollständiger Vergleich aller Modelle:
- Metriken auf Val- und Test-Set
- Feature Importance (XGBoost & LightGBM)
- Tiefergehende Modellanalyse
- Fehleranalyse pro Store
- Residuenanalyse des besten Modells

## 0 · Imports & Setup

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.append(os.path.abspath('../03_src'))
from config import FINAL, TARGET_COL, FEATURE_COLS, EXOG_COLS, STAT_EXOG, HIST_EXOG, TRAIN_END, VAL_END, LOOKBACK, HORIZON, MODELL_ORDER, MODELL_COLORS, PRED_COLS, RESULTS, FILE_NAMES
from utilis import run_sarimax, run_prophet, run_xgb, run_lgbm, to_nf_format, compute_metrics, load_preds

sns.set_style('whitegrid')

print('Setup ✓')

Setup ✓


## 1 · Daten laden

In [5]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train     = df.filter(pl.col('date') <= TRAIN_END)
val       = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test      = df.filter(pl.col('date') > VAL_END)
train_val = df.filter(pl.col('date') <= VAL_END)

stores = sorted(df['store_nbr'].unique().to_list())

print(f'Train:     {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:       {val.shape}    {val["date"].min()} → {val["date"].max()}')
print(f'Test:      {test.shape}   {test["date"].min()} → {test["date"].max()}')
print(f'Stores:    {len(stores)}')

Train:     (70114, 37)  2013-01-29 → 2016-12-31
Val:       (7965, 37)    2017-01-01 → 2017-05-31
Test:      (4104, 37)   2017-06-01 → 2017-08-15
Stores:    54


In [6]:
# Val-Vorhersagen laden
val_preds = {m: load_preds(f'val_{m.lower()}.parquet') for m in MODELL_ORDER}
available = [m for m, df_m in val_preds.items() if df_m is not None]
print(f'Verfügbare Val-Modelle: {available}')

⚠  val_xgboost.parquet nicht gefunden — wird übersprungen
⚠  val_lightgbm.parquet nicht gefunden — wird übersprungen
Verfügbare Val-Modelle: ['SARIMAX', 'Prophet', 'PatchTST', 'NHITS']


## 2 · Val-Set Metriken

In [ ]:
metrics_val = []

for modell in available:
    preds_df = val_preds[modell]
    pred_col = PRED_COLS[modell]
    merged   = preds_df.drop_nulls(subset=[pred_col, 'y_true'])
    metrics_val.append(
        compute_metrics(merged['y_true'].to_numpy(), merged[pred_col].to_numpy(), modell, 'val')
    )

metrics_val_df = pd.DataFrame(metrics_val).set_index('modell').drop(columns='split')
print('=== Val-Set Metriken ===')
print(metrics_val_df.sort_values('MAE').round(2))

## 3 · Test-Set: Re-Run aller Modelle

In [ ]:
# SARIMAX
print('SARIMAX...')
test_sarimax = run_sarimax(train_val, test, stores=stores, exog_cols=EXOG_COLS)
test_sarimax.write_parquet(RESULTS / 'test_sarimax.parquet')
print('SARIMAX ✓')

In [ ]:
# Prophet
print('Prophet...')
test_prophet = run_prophet(train_val, test, stores=stores, extra_regressors=['oil_price'])
test_prophet.write_parquet(RESULTS / 'test_prophet.parquet')
print('Prophet ✓')

In [ ]:
# XGBoost & LightGBM
X_train_val = train_val.select(FEATURE_COLS).to_numpy()
y_train_val = train_val.select(TARGET_COL).to_numpy().ravel()
X_test      = test.select(FEATURE_COLS).to_numpy()
y_test      = test.select(TARGET_COL).to_numpy().ravel()

model_xgb,  _ = run_xgb(X_train_val, y_train_val, X_test, y_test)
model_lgbm, _ = run_lgbm(X_train_val, y_train_val, X_test, y_test)

test_xgb = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_xgb', model_xgb.predict(X_test).astype(float)))
)
test_lgbm = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_lgbm', model_lgbm.predict(X_test).astype(float)))
)
test_xgb.write_parquet(RESULTS / 'test_xgb.parquet')
test_lgbm.write_parquet(RESULTS / 'test_lgbm.parquet')
print('XGBoost & LightGBM ✓')

In [ ]:
# PatchTST & NHITS
import torch
torch.set_num_threads(1)
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, NHITS

train_val_nf = to_nf_format(train_val)
static_df = (
    train_val.select(['store_nbr'] + STAT_EXOG)
             .unique(subset=['store_nbr']).sort('store_nbr')
             .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
             .select(['unique_id'] + STAT_EXOG)
             .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
             .to_pandas()
)

def nf_to_polars(preds_pd, model_col, pred_col, ref_df):
    return (
        pl.from_pandas(preds_pd.reset_index())
        .rename({'unique_id': 'store_nbr', 'ds': 'date', model_col: pred_col})
        .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
        .join(ref_df.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
              on=['store_nbr', 'date'], how='left')
    )

# PatchTST
print('PatchTST...')
nf_pt = NeuralForecast(models=[PatchTST(
    h=HORIZON, input_size=LOOKBACK, patch_len=16, stride=8,
    encoder_layers=2, n_heads=8, hidden_size=64, linear_hidden_size=128,
    dropout=0.2, fc_dropout=0.2, scaler_type='standard',
    max_steps=200, batch_size=64, learning_rate=1e-4,
    early_stop_patience_steps=10, val_check_steps=25, random_seed=42,
)], freq='D')
nf_pt.fit(df=train_val_nf[['unique_id', 'ds', 'y']], val_size=len(test['date'].unique()))
test_patchtst = nf_to_polars(nf_pt.predict(), 'PatchTST', 'pred_patchtst', test)
test_patchtst.write_parquet(RESULTS / 'test_patchtst.parquet')
print('PatchTST ✓')

In [ ]:
# NHITS
print('NHITS...')
required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG
nf_nh = NeuralForecast(models=[NHITS(
    h=HORIZON, input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG, hist_exog_list=HIST_EXOG,
    scaler_type='standard', max_steps=200, batch_size=128,
    learning_rate=1e-3, early_stop_patience_steps=5,
    val_check_steps=10, random_seed=42,
)], freq='D')
nf_nh.fit(df=train_val_nf[required_cols_nhits], static_df=static_df, val_size=len(test['date'].unique()))
test_nhits = nf_to_polars(nf_nh.predict(), 'NHITS', 'pred_nhits', test)
test_nhits.write_parquet(RESULTS / 'test_nhits.parquet')
print('NHITS ✓')

## 4 · Test-Set Metriken

In [ ]:
test_preds = {m: load_preds(f'test_{m.lower()}.parquet') for m in MODELL_ORDER}

metrics_test = []
for modell, preds_df in test_preds.items():
    if preds_df is None:
        continue
    pred_col = PRED_COLS[modell]
    merged = preds_df.drop_nulls(subset=[pred_col, 'y_true'])
    metrics_test.append(
        compute_metrics(merged['y_true'].to_numpy(), merged[pred_col].to_numpy(), modell, 'test')
    )

metrics_test_df = pd.DataFrame(metrics_test).set_index('modell').drop(columns='split')
print('=== Test-Set Metriken ===')
print(metrics_test_df.sort_values('MAE').round(2))

## 5 · Metriken-Vergleich: Val vs. Test

In [ ]:
modell_order = [m for m in MODELL_ORDER
                if m in metrics_val_df.index and m in metrics_test_df.index]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    x = np.arange(len(modell_order))
    val_vals  = metrics_val_df.reindex(modell_order)[metric].values
    test_vals = metrics_test_df.reindex(modell_order)[metric].values
    ax.bar(x - 0.2, val_vals,  width=0.35, color='steelblue', alpha=0.85, label='Val')
    ax.bar(x + 0.2, test_vals, width=0.35, color='tomato',    alpha=0.85, label='Test')
    ax.set_xticks(x)
    ax.set_xticklabels(modell_order, rotation=30, ha='right')
    ax.set_title(metric, fontsize=13)
    ax.legend()

fig.suptitle('Modellvergleich — Val vs. Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'model_comparison_val_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Tiefergehende Analyse: XGBoost & LightGBM

XGBoost und LightGBM erzielten die besten Ergebnisse. Im Folgenden analysieren wir welche Features die Vorhersagekraft treiben.

In [ ]:
# Feature Importance: Top 20
feat_names = FEATURE_COLS

imp_xgb  = pd.Series(model_xgb.feature_importances_,  index=feat_names).sort_values(ascending=False)
imp_lgbm = pd.Series(model_lgbm.feature_importances_, index=feat_names).sort_values(ascending=False)

top_n = 20
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, imp, label, color in zip(
    axes,
    [imp_xgb, imp_lgbm],
    ['XGBoost', 'LightGBM'],
    ['seagreen', 'purple']
):
    imp.head(top_n).sort_values().plot(kind='barh', ax=ax, color=color, alpha=0.85)
    ax.set_title(f'Feature Importance — {label} (Top {top_n})', fontsize=13)
    ax.set_xlabel('Importance')
    ax.axvline(imp.head(top_n).mean(), color='red', linestyle='--', linewidth=1, label='Ø')
    ax.legend()

plt.tight_layout()
plt.savefig(RESULTS / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature-Vergleich: XGBoost vs. LightGBM (normalisiert)
imp_xgb_norm  = imp_xgb  / imp_xgb.sum()
imp_lgbm_norm = imp_lgbm / imp_lgbm.sum()

compare = pd.DataFrame({
    'XGBoost':  imp_xgb_norm,
    'LightGBM': imp_lgbm_norm,
}).dropna().sort_values('XGBoost', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
compare.sort_values('XGBoost').plot(kind='barh', ax=ax,
    color=['seagreen', 'purple'], alpha=0.8, width=0.7)
ax.set_title('Feature Importance Vergleich — XGBoost vs. LightGBM (normalisiert, Top 15)', fontsize=13)
ax.set_xlabel('Relative Importance')
plt.tight_layout()
plt.savefig(RESULTS / 'feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Features XGBoost:')
print(imp_xgb_norm.head(10).round(4).to_string())
print('\nTop 10 Features LightGBM:')
print(imp_lgbm_norm.head(10).round(4).to_string())

In [ ]:
# Lernkurven: Train vs. Val Loss über Boosting-Runden
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# XGBoost
if hasattr(model_xgb, 'evals_result_'):
    evals = model_xgb.evals_result_
    for key, vals in evals.items():
        for metric, scores in vals.items():
            axes[0].plot(scores, label=key)
    axes[0].set_title('XGBoost — Lernkurve', fontsize=12)
    axes[0].set_xlabel('Boosting Runde')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

# LightGBM
if hasattr(model_lgbm, 'evals_result_'):
    evals = model_lgbm.evals_result_
    for key, vals in evals.items():
        for metric, scores in vals.items():
            axes[1].plot(scores, label=key)
    axes[1].set_title('LightGBM — Lernkurve', fontsize=12)
    axes[1].set_xlabel('Boosting Runde')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Fehleranalyse pro Store (Test-Set)

In [ ]:
def mae_per_store(preds_df, pred_col):
    return (
        preds_df
        .with_columns((pl.col(pred_col) - pl.col('y_true')).abs().alias('abs_error'))
        .group_by('store_nbr')
        .agg(pl.col('abs_error').mean().alias('mae'))
        .sort('store_nbr')
    )

store_mae = None
for modell, preds_df in test_preds.items():
    if preds_df is None:
        continue
    s = mae_per_store(preds_df, PRED_COLS[modell]).rename({'mae': modell})
    store_mae = s if store_mae is None else store_mae.join(s, on='store_nbr', how='inner')

store_mae_pd = store_mae.to_pandas().set_index('store_nbr')

fig, ax = plt.subplots(figsize=(16, 5))
store_mae_pd.plot(kind='bar', ax=ax, width=0.75, alpha=0.85,
                  color=[MODELL_COLORS[m] for m in store_mae_pd.columns])
ax.set_title('MAE pro Store — Test-Set', fontsize=13)
ax.set_xlabel('Store')
ax.set_ylabel('MAE')
ax.legend(title='Modell', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(RESULTS / 'mae_per_store.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSchlechteste 10 Stores (Ø MAE über alle Modelle):')
print(store_mae_pd.mean(axis=1).sort_values(ascending=False).head(10).round(1))

## 8 · Zeitverlauf: Forecast vs. Realität (Store 1)

In [ ]:
store_id = 1

fig, ax = plt.subplots(figsize=(15, 5))

ist = test.filter(pl.col('store_nbr') == store_id).sort('date')
ax.plot(ist['date'].to_list(), ist[TARGET_COL].to_list(),
        color='black', linewidth=2, label='Ist-Wert', zorder=5)

for modell, preds_df in test_preds.items():
    if preds_df is None:
        continue
    s = preds_df.filter(pl.col('store_nbr') == store_id).sort('date')
    if s.is_empty():
        continue
    ax.plot(s['date'].to_list(), s[PRED_COLS[modell]].to_list(),
            color=MODELL_COLORS[modell], linewidth=1.2,
            linestyle='--', label=modell, alpha=0.85)

ax.set_title(f'Forecast vs. Realität — Store {store_id} (Test-Set)', fontsize=13)
ax.set_ylabel('Transaktionen')
ax.set_xlabel('Datum')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RESULTS / f'forecast_store{store_id}.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Residuenanalyse — bestes Modell

In [ ]:
best_model = metrics_test_df['MAE'].idxmin()
print(f'Bestes Modell nach MAE auf Test-Set: {best_model}')

best_df   = test_preds[best_model]
best_col  = PRED_COLS[best_model]
best_pd   = (
    best_df
    .with_columns((pl.col(best_col) - pl.col('y_true')).alias('residual'))
    .drop_nulls(subset=['residual'])
    .to_pandas()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogramm
axes[0].hist(best_pd['residual'], bins=60, color=MODELL_COLORS[best_model],
             edgecolor='none', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[0].set_title(f'Residuen-Verteilung ({best_model})', fontsize=12)
axes[0].set_xlabel('Residual (Pred − True)')
axes[0].set_ylabel('Häufigkeit')

# Scatter: Pred vs. True
axes[1].scatter(best_pd['y_true'], best_pd[best_col],
                alpha=0.15, s=5, color=MODELL_COLORS[best_model])
lims = [best_pd['y_true'].min(), best_pd['y_true'].max()]
axes[1].plot(lims, lims, 'r--', linewidth=1.2)
axes[1].set_title('Predicted vs. True', fontsize=12)
axes[1].set_xlabel('True')
axes[1].set_ylabel('Predicted')

# Residuen über Zeit (Store 1)
store1 = best_pd[best_pd['store_nbr'] == store_id]
axes[2].plot(store1['date'], store1['residual'],
             linewidth=0.9, color=MODELL_COLORS[best_model])
axes[2].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[2].set_title(f'Residuen über Zeit — Store {store_id}', fontsize=12)
axes[2].set_ylabel('Residual')
plt.setp(axes[2].get_xticklabels(), rotation=30)

plt.tight_layout()
plt.savefig(RESULTS / f'residuals_{best_model.lower()}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Bias (Ø Residual): {best_pd["residual"].mean():.2f}')
print(f'Std Residual:      {best_pd["residual"].std():.2f}')

## 10 · Finale Zusammenfassung

In [ ]:
print('=' * 65)
print('MODELLVERGLEICH — FINALE ERGEBNISSE')
print('=' * 65)

print('\nVAL-SET (sortiert nach MAE):')
print(metrics_val_df.sort_values('MAE').round(2).to_string())

print('\nTEST-SET (sortiert nach MAE):')
print(metrics_test_df.sort_values('MAE').round(2).to_string())

print(f'\n→ Bestes Modell (Test MAE): {best_model}')

# Speichern
pl.from_pandas(metrics_val_df.reset_index()).write_parquet(RESULTS / 'metrics_val.parquet')
pl.from_pandas(metrics_test_df.reset_index()).write_parquet(RESULTS / 'metrics_test.parquet')
print('Metriken gespeichert ✓')